<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/14_Deep_Learning/01_MLP_Carbon_Flux.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 14: Deep Learning Foundations
## Part 01: Multilayer Perceptron (MLP) for Tabular Carbon Flux Anomaly Detection

### Objective
Transition from traditional machine learning (SVM) into deep representation learning. Using high-frequency meteorological and eddy-covariance sensor data (FLUXNET / Ameriflux tier observations), we construct a feed-forward Multilayer Perceptron to classify localized extreme Net Ecosystem Exchange (NEE) / carbon flux anomalies.

### Key Milestones
1. Ingest raw continuous atmospheric sensor observations (temperature, radiation, vapor pressure deficit, soil metrics).
2. Engineer a binary anomaly target representing extreme respiration / emission events (upper 12th percentile).
3. Implement non-linear scaling tailored for neural gradient descent.
4. Construct, train, and validate a Dense Neural Network using TensorFlow/Keras.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Create project subdirectories inside Colab's workspace
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

# Verify environment
print(f"TensorFlow Version: {tf.__version__}")
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"⚡ GPU Detected: {gpu_devices}")
else:
    print("ℹ️ Running on CPU backend.")

# Reproducibility seed
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

TensorFlow Version: 2.21.0
ℹ️ Running on CPU backend.


In [3]:
#Synthetic FLUXNET Sensor Generation
def generate_fluxnet_sensor_stream(n_samples=18000, random_state=42):
    rng = np.random.default_rng(random_state)

    # 1. Base micro-meteorological variables
    ta = rng.normal(loc=18.5, scale=8.2, size=n_samples)
    sw_in = np.clip(rng.exponential(scale=180.0, size=n_samples), 0, 1100)
    vpd = np.clip(0.12 * ta + rng.exponential(scale=5.0, size=n_samples), 0.1, 45.0)
    ustar = np.clip(rng.gamma(shape=2.5, scale=0.18, size=n_samples), 0.02, 2.5)
    swc = np.clip(rng.beta(a=2.0, b=5.0, size=n_samples) * 100, 2.0, 55.0)
    ts = ta * 0.85 + rng.normal(0, 2.2, size=n_samples)

    # 2. Energy balance variables
    le = np.clip(0.45 * sw_in - 2.8 * vpd + rng.normal(0, 25, size=n_samples), -30, 650)
    h = np.clip(0.35 * sw_in + 3.1 * (ta - ts) + rng.normal(0, 30, size=n_samples), -40, 500)

    # 3. Fixed non-linear stress response (Clipping negative temps to 0)
    stress_index = (
        0.04 * (np.maximum(ts, 0) ** 1.8)
        + 0.15 * (vpd ** 1.4)
        - 0.08 * swc
        + 1.5 * np.sin(ta / 5.0)
        + rng.normal(0, 3.5, size=n_samples)
    )

    # 4. Target Labeling
    threshold = np.percentile(stress_index, 88)
    flux_anomaly = (stress_index >= threshold).astype(int)

    df = pd.DataFrame({
        'Air_Temp': ta,
        'Shortwave_Radiation': sw_in,
        'VPD': vpd,
        'Friction_Velocity': ustar,
        'Latent_Heat': le,
        'Sensible_Heat': h,
        'Soil_Water_Content': swc,
        'Soil_Temp': ts,
        'Flux_Anomaly': flux_anomaly
    })

    return df

df_flux = generate_fluxnet_sensor_stream()
data_path = "data/raw_carbon_flux_sensors.csv"
df_flux.to_csv(data_path, index=False)

print(f"✅ Sensor stream synthesized and saved to '{data_path}'")
print(f"Dataset Shape: {df_flux.shape}")
print(f"Class Distribution:\n{df_flux['Flux_Anomaly'].value_counts(normalize=True).round(4)}")
df_flux.head(5)

✅ Sensor stream synthesized and saved to 'data/raw_carbon_flux_sensors.csv'
Dataset Shape: (18000, 9)
Class Distribution:
Flux_Anomaly
0    0.88
1    0.12
Name: proportion, dtype: float64


,Air_Temp,Shortwave_Radiation,VPD,Friction_Velocity,Latent_Heat,Sensible_Heat,Soil_Water_Content,Soil_Temp,Flux_Anomaly
0,20.998680,184.112835,3.021395,0.903374,69.720089,79.049116,21.769193,19.807731,0
1,9.972130,152.418856,2.197495,0.308442,52.920126,25.376272,15.271716,5.564608,0
2,24.653700,239.481033,15.606681,0.162800,108.448287,106.866312,24.434922,22.074876,1
3,26.212631,563.896980,5.418580,0.472831,250.815116,228.749107,24.513004,23.685713,0
4,2.501511,214.367137,2.355494,1.086367,106.568671,79.148486,50.011076,3.552431,0


### Sensor Data Ingestion & Class Imbalance Analysis

The synthetic FLUXNET stream has successfully generated **18,000 temporal observations** across 8 meteorological features and 1 binary target.

**Key structural implications for our Deep Learning architecture:**
* **Feature Complexity:** The dataset contains continuous, multi-scale variables representing both energy balance (Latent/Sensible Heat) and physical atmospheric stress (VPD, Soil Water Content, Friction Velocity). Neural networks are highly sensitive to varying scales, making rigorous normalization mandatory in the next step.
* **Severe Class Imbalance:** The target variable (`Flux_Anomaly`) is heavily skewed, with 88% nominal observations and only 12% representing extreme carbon emission spikes.
* **Evaluation Shift:** Because of this 88/12 split, standard model accuracy is a mathematically invalid metric. A "lazy" network could predict 0 every time and still achieve 88% accuracy. We will construct our MLP to optimize for and be evaluated on **Precision, Recall, and the ROC-AUC score**.

### Part 02: Tensor Preparation & Non-Linear Scaling

Before feeding data into a neural network, we must restructure it into mathematical tensors. This process involves two critical steps:

1. **Strategic Partitioning:** We split the data into three distinct sets:
   * **Training Set (70%):** The data the network uses to calculate gradients and update its weights.
   * **Validation Set (15%):** Unseen data used actively during training to monitor for overfitting and trigger early stopping mechanisms.
   * **Test Set (15%):** The final, completely blind holdout set used to evaluate the ultimate model performance.
2. **Robust Normalization:** Neural networks calculate weight updates using gradient descent. If features are on vastly different scales (e.g., incoming solar radiation in the 1000s vs. friction velocity in decimals), the gradients will explode or vanish, causing the mathematical engine to fail. Furthermore, because we are specifically looking for extreme anomaly spikes, standard scaling would squash and destroy our outlier signals. Instead, we use a **RobustScaler**, which scales features using the median and interquartile range, preserving the natural impact of true physical anomalies.

In [5]:
# 1. Isolate features (X) and target (y)
X = df_flux.drop(columns=['Flux_Anomaly'])
y = df_flux['Flux_Anomaly']

# 2. First split: 70% Train, 30% Temporary (Val/Test)
# We use stratify=y to ensure the 12% anomaly ratio is perfectly maintained across all splits
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

# 3. Second split: Divide the 30% equally into 15% Validation and 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

# 4. Initialize and fit the RobustScaler STRICTLY on the Training set to prevent data leakage
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 5. Convert NumPy arrays to explicit TensorFlow float32 Tensors
# This ensures strict type compatibility for optimized edge deployment (like TFLite)
X_train_tf = tf.convert_to_tensor(X_train_scaled, dtype=tf.float32)
y_train_tf = tf.convert_to_tensor(y_train.values, dtype=tf.float32)

X_val_tf = tf.convert_to_tensor(X_val_scaled, dtype=tf.float32)
y_val_tf = tf.convert_to_tensor(y_val.values, dtype=tf.float32)

X_test_tf = tf.convert_to_tensor(X_test_scaled, dtype=tf.float32)
y_test_tf = tf.convert_to_tensor(y_test.values, dtype=tf.float32)

print("✅ Tensor Conversion and Scaling Complete.")
print("-" * 45)
print(f"Training Tensors:   X: {X_train_tf.shape} | y: {y_train_tf.shape}")
print(f"Validation Tensors: X: {X_val_tf.shape}  | y: {y_val_tf.shape}")
print(f"Testing Tensors:    X: {X_test_tf.shape}  | y: {y_test_tf.shape}")

✅ Tensor Conversion and Scaling Complete.
---------------------------------------------
Training Tensors:   X: (12600, 8) | y: (12600,)
Validation Tensors: X: (2700, 8)  | y: (2700,)
Testing Tensors:    X: (2700, 8)  | y: (2700,)


### Part 03: Architecting the Neural Network

With our tensors prepared, we construct the Multilayer Perceptron. To ensure the architecture remains strictly compatible with edge deployment on Android and iOS devices (via TFLite), we will build a lightweight but deep sequential model using standard, fully supported Keras layers.

**The Architecture:**
1. **Input & Hidden Layers:** We use `Dense` (fully connected) layers. The first layer expands the 8 input features into a higher-dimensional space (32 neurons) to discover complex interactions. We use the **ReLU (Rectified Linear Unit)** activation function, which excels at handling non-linear data without vanishing gradients.
2. **Regularization (Dropout):** Atmospheric data is inherently noisy. To prevent the network from simply memorizing the training noise (overfitting), we insert `Dropout` layers. These randomly deactivate a percentage of neurons during training, forcing the network to learn robust, generalized patterns.
3. **Output Layer:** A single neuron using the **Sigmoid** activation function, which mathematically squashes the final output into a clean probability score between 0.0 and 1.0 (representing the likelihood of a carbon flux anomaly).
4. **Compilation Engine:** We compile the model using the **Adam** optimizer (adaptive learning rate) and **Binary Crossentropy** for the loss function. Because of our 88/12 class imbalance, we instruct the engine to track AUC (Area Under the Curve), Precision, and Recall rather than basic accuracy.

In [6]:
# 1. Define the input shape dynamically based on our tensor
input_dim = X_train_tf.shape[1]

# 2. Architect the Sequential Model
model = keras.Sequential([
    # Input Layer & First Hidden Layer
    layers.Dense(32, activation='relu', input_shape=(input_dim,)),
    layers.Dropout(0.2), # Randomly drops 20% of connections to prevent overfitting

    # Second Hidden Layer (compressing the representation)
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),

    # Third Hidden Layer
    layers.Dense(8, activation='relu'),

    # Output Layer: Single node with Sigmoid for binary probability (0 to 1)
    layers.Dense(1, activation='sigmoid')
])

# 3. Define metrics tailored for imbalanced classification
metrics = [
    keras.metrics.AUC(name='auc'),
    keras.metrics.Precision(name='precision'),
    keras.metrics.Recall(name='recall')
]

# 4. Compile the engine
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=metrics
)

# Display the architectural map
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 961 (3.75 KB)

 Trainable params: 961 (3.75 KB)

 Non-trainable params: 0 (0.00 B)